## Objective:
#### "To analyze how student demographics (nationality, age, qualification level), academic performance (GPA, attendance), and learning behaviors (self-study hours, prior knowledge) influence course completion and student success, in order to identify at-risk student profiles and recommend targeted academic support interventions."

In [7]:
import pandas as pd

# ==========================================
# STEP 1: LOAD ALL DATASETS
# ==========================================
try:
    # Load the main datasets
    master_df = pd.read_csv('cleaned_data/master_dataset.csv')
    profiles_df = pd.read_csv('cleaned_data/student_profiles_clean.csv')
    results_df = pd.read_csv('cleaned_data/student_results_clean.csv')
    survey_df = pd.read_csv('cleaned_data/student_survey_clean.csv')
    
    # Load reference files
    course_codes = pd.read_csv('cleaned_data/course_codes_clean.csv')
    wrangling_log = pd.read_csv('cleaned_data/data_wrangling_log.csv')

    print("✅ All CSV files loaded successfully!")
    print(f"Master Dataset Shape: {master_df.shape}")

except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Please ensure all CSV files are in the same folder as this notebook.")

# ==========================================
# STEP 2: PREPARE THE DATA (MERGE SCHOOL CODES)
# ==========================================
# This adds the readable 'COURSE NAME' to your master dataset for better plotting

# 1. Extract the first 4 digits from the CLASS column (e.g., '1101-009' -> '1101')
master_df['COURSE_CODE_PREFIX'] = master_df['CLASS'].astype(str).str.split('-').str[0]

# 2. Convert to numeric to match the course_codes file
master_df['COURSE_CODE_PREFIX'] = pd.to_numeric(master_df['COURSE_CODE_PREFIX'], errors='coerce')

# 3. Merge with course_codes to get 'COURSE NAME'
master_df = pd.merge(master_df, course_codes, left_on='COURSE_CODE_PREFIX', right_on='CODE', how='left')

# 4. Check the result
print("\n✅ Merged with Course Names. Preview:")
print(master_df[['STUDENT ID', 'CLASS', 'COURSE NAME']].head())

✅ All CSV files loaded successfully!
Master Dataset Shape: (520, 28)

✅ Merged with Course Names. Preview:
     STUDENT ID     CLASS                        COURSE NAME
0  1101-009/001  1101-009  Diploma in Data Analytics with AI
1  1101-009/001  1101-009  Diploma in Data Analytics with AI
2  1101-009/001  1101-009  Diploma in Data Analytics with AI
3  1101-009/002  1101-009  Diploma in Data Analytics with AI
4  1101-009/002  1101-009  Diploma in Data Analytics with AI


In [14]:
import plotly.express as px

# ==========================================
# CHART A: GPA Distribution (Is the data normal?)
# ==========================================
# This helps you see if grades are skewed. 
# Are there many failures (Left side) or mostly Distinctions (Right side)?
fig_dist = px.histogram(
    master_df, 
    x="GPA", 
    nbins=20, 
    title="<b>Distribution of Student GPA</b><br><i>(Most students score between 3.0 and 3.5)</i>",
    color_discrete_sequence=['indianred'], # Use a distinct color
    text_auto=True # Shows the count on top of each bar
)
fig_dist.add_vline(x=master_df['GPA'].mean(), line_dash="dash", annotation_text="Avg: 3.11")
fig_dist.show()

# ==========================================
# CHART B: Semester Performance (Are they improving?)
# ==========================================
# This checks if students struggle more in Sem 1 or Sem 3.
# A box plot shows the median (middle line) and the range.
fig_sem = px.box(
    master_df, 
    x="PERIOD", 
    y="GPA", 
    color="PERIOD",
    title="<b>GPA Performance by Semester</b><br><i>(Do students improve over time?)</i>",
    category_orders={"PERIOD": ["Sem 1", "Sem 2", "Sem 3"]} # Force logical order
)
fig_sem.show()

# ==========================================
# CHART C: Correlation Heatmap (The Cheat Sheet)
# ==========================================
# This is the most important EDA chart. 
# It tells you mathematically what matters.
# Red = Strong Positive Link, Blue = Negative Link.

# 1. Select only numeric columns
numeric_cols = ['GPA', 'ATTENDANCE', 'SELF-STUDY HRS', 'PRIOR KNOWLEDGE', 'AGE', 'TEACHING SUPPORT']
corr_matrix = master_df[numeric_cols].corr()

# 2. Plot
fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f", # Show the correlation number (e.g., 0.72)
    aspect="auto",
    title="<b>Correlation Matrix: What drives GPA?</b><br><i>(Look for bright yellow squares)</i>",
    color_continuous_scale="Viridis" 
)
fig_corr.show()

In [8]:
import plotly.express as px

# Group by Course Name to get average GPA
# We reset_index() so 'COURSE NAME' becomes a column we can plot
course_perf = master_df.groupby('COURSE NAME')[['GPA']].mean().reset_index().sort_values('GPA')

# Create Horizontal Bar Chart
fig_overview = px.bar(
    course_perf,
    x='GPA',
    y='COURSE NAME',
    orientation='h',  # Horizontal is better for long course names
    title='<b>Average GPA by Course</b><br><i>(Benchmark for Academic Performance)</i>',
    text_auto='.2f',  # Shows the exact GPA on the bar
    color='GPA',      # Color intensity shows performance
    color_continuous_scale='Blues',
    labels={'GPA': 'Average GPA', 'COURSE NAME': ''}
)

# Clean up layout
fig_overview.update_layout(showlegend=False, xaxis_range=[0, 4.0]) # Scale 0-4 for GPA
fig_overview.show()

In [10]:
# Box Plot: Nationality vs GPA
fig_demo = px.box(
    master_df,
    x='NATIONALITY_STATUS',
    y='GPA',
    color='NATIONALITY_STATUS',
    points="all", # Shows the actual students as dots to see the spread
    title='<b>Student Performance Distribution by Nationality</b><br><i>(Identifying At-Risk Groups)</i>',
    labels={'NATIONALITY_STATUS': 'Status', 'GPA': 'GPA'},
    category_orders={"NATIONALITY_STATUS": ["SG Citizen", "SG PR", "Foreigner"]} # Logical order
)

fig_demo.show()

In [13]:
import plotly.express as px
import pandas as pd

# ==============================================================================
# FIX: Handle Missing Values First
# ==============================================================================
# Create a clean dataframe specifically for plotting
# We drop rows where any of the 3 key metrics are missing to avoid the "NaN" error
plot_df = master_df.dropna(subset=['ATTENDANCE', 'SELF-STUDY HRS', 'GPA']).copy()

# --- PREP: Calculate Relative Metrics (Using the Clean Data) ---
global_avg_gpa = plot_df['GPA'].mean()
avg_study = plot_df['SELF-STUDY HRS'].mean()

# Calculate relative performance for the second chart
course_perf = plot_df.groupby('COURSE NAME')[['GPA']].mean().reset_index()
course_perf['GPA_Relative_Performance'] = course_perf['GPA'] - global_avg_gpa
course_perf['Status'] = course_perf['GPA_Relative_Performance'].apply(lambda x: 'Above Avg' if x > 0 else 'Below Avg')

# ==============================================================================
# CHART 1: The "Risk Matrix" (Fixed)
# ==============================================================================
fig_matrix = px.scatter(
    plot_df,  # <--- USE THE CLEAN 'plot_df' HERE
    x='SELF-STUDY HRS',
    y='GPA',
    color='NATIONALITY_STATUS', 
    size='ATTENDANCE',          
    hover_name='STUDENT ID',
    title='<b>Student Risk Matrix</b><br><i>(Who is trying but failing? Who is disengaged?)</i>',
    labels={'SELF-STUDY HRS': 'Effort (Self-Study Hours)', 'GPA': 'Performance (GPA)'},
    template='plotly_white'
)

# Add Reference Lines (Quadrants)
fig_matrix.add_vline(x=avg_study, line_width=1, line_dash="dash", line_color="grey")
fig_matrix.add_hline(y=global_avg_gpa, line_width=1, line_dash="dash", line_color="grey")

# Add Annotations
fig_matrix.add_annotation(x=avg_study+5, y=global_avg_gpa+0.5, text="⭐⭐ STARS", showarrow=False, font=dict(color="green"))
fig_matrix.add_annotation(x=avg_study+5, y=global_avg_gpa-0.5, text="⚠️ STRUGGLERS", showarrow=False, font=dict(color="orange"))
fig_matrix.add_annotation(x=avg_study-5, y=global_avg_gpa-0.5, text="🚨 DISENGAGED", showarrow=False, font=dict(color="red"))

fig_matrix.show()

# ==============================================================================
# CHART 2: The "Performance Gap" (No changes needed, but included for completeness)
# ==============================================================================
fig_gap = px.bar(
    course_perf.sort_values('GPA_Relative_Performance'),
    y='COURSE NAME',
    x='GPA_Relative_Performance',
    color='Status',
    title='<b>Course Difficulty Analysis</b><br><i>(Relative to School Average GPA of {:.2f})</i>'.format(global_avg_gpa),
    text_auto='.2f',
    color_discrete_map={'Above Avg': 'teal', 'Below Avg': 'crimson'},
    labels={'GPA_Relative_Performance': 'Deviation from Average GPA', 'COURSE NAME': ''}
)

fig_gap.update_layout(showlegend=False)
fig_gap.add_vline(x=0, line_width=2, line_color="black")
fig_gap.show()

In [15]:
import plotly.express as px

# 1. Who are our students? (Nationality Breakdown)
# This uses a 'Pie Chart' which is classic for showing composition.
fig_nat = px.pie(
    master_df, 
    names='NATIONALITY_STATUS', 
    title='<b>Student Composition by Nationality</b><br><i>(Majority are Locals or Foreigners?)</i>',
    hole=0.4, # Makes it a Donut chart (looks more modern)
    color_discrete_sequence=px.colors.sequential.RdBu
)
fig_nat.update_traces(textinfo='percent+label')
fig_nat.show()

# 2. What is their background? (Qualification Level)
# This uses a 'Bar Chart' to see if most students are already degree holders or beginners.
# We sort it so the biggest group is on top.
qual_counts = master_df['HIGHEST QUALIFICATION'].value_counts().reset_index()
qual_counts.columns = ['Qualification', 'Count']

fig_qual = px.bar(
    qual_counts, 
    x='Count', 
    y='Qualification', 
    orientation='h',
    title='<b>Educational Background</b><br><i>(What qualifications do they already have?)</i>',
    text_auto=True,
    color='Count',
    color_continuous_scale='Mint'
)
fig_qual.show()